In [1]:
# 1. Install 'pak' - the modern, foolproof R package manager
install.packages("pak", repos = "https://cran.rstudio.com/")

# 2. Let pak handle the Bioconductor installation.
# It automatically maps system dependencies and uses fast Linux binaries.
pak::pkg_install(c("ensembldb", "AnnotationHub"))

# 3. Setup your project directories
dir.create("/content/compbioass", recursive = TRUE, showWarnings = FALSE)
setwd("/content/compbioass")

# 4. Load the newly installed packages
library(ensembldb)
library(AnnotationHub)

# 5. Connect to the database
setAnnotationHubOption("CACHE", "/content/compbioass/AH_cache")
ah <- AnnotationHub()
edb <- ah[["AH119325"]]

print("--- PAK SUCCESSFULLY INSTALLED AND LOADED EVERYTHING! ---")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

ℹ Loading metadata database

✔ Loading metadata database ... done



 

→ Will install 19 packages.

→ All 19 packages (553.40 kB) are cached.

+ AnnotationDbi          1.74.0 [bld]
+ AnnotationFilter       1.36.0 [bld]
+ AnnotationHub          4.2.0  [bld][cmp]
+ BiocBaseUtils          1.14.0 [bld]
+ BiocFileCache          3.2.0  [bld]
+ BiocVersion            3.23.1 [bld]
+ Biostrings             2.80.0 [bld][cmp]
+ cigarillo              1.2.0  [bld][cmp]
+ dbplyr                 2.5.2  [bld]
+ DelayedArray           0.38.1 [bld]
+ ensembldb              2.36.0 [bld]
+ GenomicAlignments      1.48.0 [bld][cmp]
+ GenomicFeatures        1.64.0 [bld][cmp]
+ GenomicRanges          1.64.0 [bld]
+ KEGGREST               1.52.0 [bld]
+ Rsamtools              2.28.0 [bld][cmp] + ✔ make
+ rtracklayer            1.72.0 [bld][cmp]
+ SparseArray            1.12.2 [bld][cmp]
+ SummarizedExperiment   1.42.0 [bld]

✔

downloading 1 resources

retrieving 1 resource



loading from cache



[1] "--- PAK SUCCESSFULLY INSTALLED AND LOADED EVERYTHING! ---"


In [2]:
# ---------------------------------------------------------
# Define your specific target gene
# ---------------------------------------------------------
my_gene <- "RYR2"

# Fetch all relevant transcripts and protein isoforms
gene_transcripts <- transcripts(edb, filter = ~ gene_name == my_gene)
gene_proteins <- proteins(edb, filter = ~ gene_name == my_gene)
protein_ids <- na.omit(gene_proteins$protein_id)

cat("\n--- Mapping all protein isoforms for:", my_gene, "---\n")

# Main execution loop mapping amino acids to chromosomal positions
mapping_results <- list()
for (pid in protein_ids) {
  prot_info <- proteins(edb, filter = ~ protein_id == pid)
  prot_length <- nchar(prot_info$protein_sequence)

  aa_range <- IRanges(start = 1, end = prot_length)
  names(aa_range) <- pid

  mapped_coords <- proteinToGenome(aa_range, edb)
  mapping_results[[pid]] <- as.data.frame(mapped_coords[[1]])
}

# Combine into a clean final data table
final_map_df <- do.call(rbind, mapping_results)

# Save the final file to your current Colab cloud path
output_filename <- paste0(my_gene, "_protein_to_genome_map.tsv")
write.table(final_map_df, file = output_filename,
            sep = "\t", row.names = FALSE, quote = FALSE)

cat("\n--- SUCCESS! ---\n")
cat("Mapping complete! Your submission file is saved as:", output_filename, "\n")


--- Mapping all protein isoforms for: RYR2 ---


Fetching CDS for 1 proteins ... 
1 found

Checking CDS and protein sequence lengths ... 
1/1 OK

Fetching CDS for 1 proteins ... 
1 found

Checking CDS and protein sequence lengths ... 
1/1 OK

Fetching CDS for 1 proteins ... 
1 found

Checking CDS and protein sequence lengths ... 
1/1 OK

Fetching CDS for 1 proteins ... 
1 found

Checking CDS and protein sequence lengths ... 
1/1 OK

Fetching CDS for 1 proteins ... 
1 found

Checking CDS and protein sequence lengths ... 
1/1 OK

Fetching CDS for 1 proteins ... 
1 found

Checking CDS and protein sequence lengths ... 
1/1 OK

Fetching CDS for 1 proteins ... 
1 found

Checking CDS and protein sequence lengths ... 
1/1 OK

Fetching CDS for 1 proteins ... 
1 found

Checking CDS and protein sequence lengths ... 
1/1 OK

Fetching CDS for 1 proteins ... 
1 found

Checking CDS and protein sequence lengths ... 
1/1 OK

Fetching CDS for 1 proteins ... 
1 found

Checking CDS and protein sequence lengths ... 
1/1 OK




--- SUCCESS! ---
Mapping complete! Your submission file is saved as: RYR2_protein_to_genome_map.tsv 
